In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp  

In [2]:
P=3
alpha=0.01
sigma=3.5e6
mur=4
mu0=4*np.pi*1e-7

In [3]:
# Exact poles, amp for conducting permable sphere
from scipy.optimize import fsolve
def exactpoleres(sigma,mu,alpha,N):
    poles=np.zeros((N))
    amp=np.zeros((N))
    beta=np.sqrt(sigma*mu)*alpha
    mur=mu/mu0

#   find solutions to fzerofun(deltan,mur) =0 
    Deltan=fsolve(fzerofun, np.pi*np.linspace(1,N,N), args=(mur))
    Deltan=np.sort(Deltan)
    #print(Deltan)
    for n in range(1,N):
        # This is for mu=mu_0
        #deltan2=np.pi**2*n**2
        # This is for general mu
        deltan2=Deltan[n-1]**2
        poles[n]=-deltan2/beta**2
        # This is for mu =mu_0
        #amp[n]=6./(deltan2)
        # For general mu
        amp[n] = 6*mur/((mur+2)*(mur-1)+deltan2)
    return poles,amp

# Residual function to help find solutions to ((mur-1)+deltan2)*(tan deltan ) - (mur-1)*deltan = 0
def fzerofun(deltan,mur):
    return ((mur-1)+deltan**2)*np.tan(deltan) - (mur-1)*deltan

In [4]:
import sympy as sym
def exact_sphere(alpha, epsilon, mur, sigma, omega):
    """
    Function to calculate the mpt for a sphere of radius alpha at radial frequency omega.
    Addapted from a Matlab function from Paul Ledger (exactsphererev5freqscan_nod.m).
    :param omega - Angular frequency rad/s
    :param epsilon - Permittivity:
    :param sigma - Conductivity S/m:
    :param mur - Relative Permeability:
    :param alpha - Sphere radius m:
    :return eig - Single unique eigenvalue of the mpt tensor for a sphere of radius alpha:
    """

    mu0 = 4 * np.pi * (1e-7)
    mu = mur * mu0
    k = np.sqrt((omega ** 2 * epsilon * sigma) + (mu * sigma * omega) * 1j)

    # Using sympy to try to avoid overflow errors.
    k *= alpha
    alpha = alpha

    js_0_kr = sym.sqrt(sym.pi / (2 * k)) * sym.besselj(1 / 2, k)
    js_1_kr = sym.sqrt(sym.pi / (2 * k)) * sym.besselj(3 / 2, k)
    js_2_kr = sym.sqrt(sym.pi / (2 * k)) * sym.besselj(5 / 2, k)
    mpt = (2 * np.pi * alpha ** 3) * (2 * (mu - mu0) * js_0_kr + (2 * mu + mu0) * js_2_kr) / (
        (mu + 2 * mu0) * js_0_kr + (mu - mu0) * js_2_kr)
    return complex(mpt.evalf())

In [5]:
#from maineigen import *
from main import *

In [ ]:
geometry="OCC_sphere_prism_4.py"
order=P
CPUs=[1,1,1,1,1,1]
Return_Dict = main(geometry=geometry,use_POD=False,use_parallel=False,use_OCC=True,start_stop=(1,8,40), MPT_Eigen=True, 
                   MPT_Eigen_From_POD=False, order=order,cpus=CPUs[order],Amp_scale=2*pi*alpha**3)
#xi,ck, sMPT, Omega =maineigen(h='coarse', order=P, curve_degree=5, geometry='OCC_sphere_prism_4.py', start_stop=(1,8,40), use_OCC=True)
sMPT=Return_Dict['TensorArray'] 
xi=Return_Dict['xi'] 
xi = xi[:,0] # We have a different format below (xi's are the same for 3 eigenmodes - some zero amplitudes)
ck=Return_Dict['ck'] 

Materials = ('object', 'air')
Updated alpha from OCC file. Alpha=0.01
OCC_sphere_prism_4.geo
[0, 1] 2 ['air', 'object']
Mesh Contains Prisms? False
N Prisms: 0, N Tets: 12484
[0, 1] 2 ['air', 'object']


/Users/pdl11/Coding/MPT-Calculator/July2026/MPT-Calculator/main.py:295: UserWarning: It looks like the main function was invoked from a jupyter notebook. 
Currently saving a .ipynb file is done by copying the most recent file in the .ipynb_checkpoints folder. 
Unless you saved the file before running the code, this may not be the correct file.
  warn('It looks like the main function was invoked from a jupyter notebook. \nCurrently saving a .ipynb file is done by copying the most recent file in the .ipynb_checkpoints folder. \nUnless you saved the file before running the code, this may not be the correct file.', stacklevel=1)


 mesh contains 12484 elements
[0, 1] 2 ['air', 'object']
Predicted unit object volume is 4.240708714123361
considering conductor element 0 2 air
considering conductor element 1 2 object
Calculated conductor volume as sum 4.240708714123361
Running Bilinear Forms Check
Mesh curve order set to:  5
K: Iteration 1: bonus_intord = 2
K: Iteration 2: bonus_intord = 4
K: Iteration 3: bonus_intord = 6
K: Iteration 4: bonus_intord = 8
K Bilinear Form Converged using order 10
C: Iteration 1: bonus_intord = 2
C: Iteration 2: bonus_intord = 4
C: Iteration 3: bonus_intord = 6
C Bilinear Form Converged using order 8


Solving Theta0: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:05<00:00,  1.68s/it]


 solved theta0 problems   


Solving Thetainf: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:04<00:00,  1.35s/it]


 solved thetainf problems   


Solving Theta1: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 107546.26it/s]


 solved theta1 problem       
 Computing coefficients  
0 / 1

In [ ]:
Omega = Return_Dict['FrequencyArray']
exactMPT=np.zeros(len(Omega),dtype=complex)
n=0
for omega in Omega:
    exactMPT[n]=exact_sphere(alpha, 0, mur, sigma, omega)
    n+=1

plt.figure(1)
plt.figure(2)
for i in range(3):
    for j in range(3):
        k= i+j*3 # index as sMPT stored as nfreq x 9
        if j == i :
            plt.figure(1)
            plt.semilogx(Omega,np.real(sMPT[:,k]),label="$(i,j)$ ("+str(i)+","+str(j)+")")
            plt.semilogx(Omega,np.real(exactMPT),'x',label="exact")
            
            plt.figure(2)
            plt.semilogx(Omega,-np.imag(exactMPT),'x',label="exact")
            plt.semilogx(Omega,np.imag(sMPT[:,k]),label="$(i,j)$ ("+str(i)+","+str(j)+")")
            

        if j < i :
            plt.figure(1)
            plt.semilogx(Omega,np.real(sMPT[:,k]),label="$(i,j)$ ("+str(i)+","+str(j)+")")
            
            plt.figure(2)
            plt.semilogx(Omega,-np.imag(sMPT[:,k]),label="$(i,j)$ ("+str(i)+","+str(j)+")")
plt.figure(1)
plt.legend()

plt.figure(2)
plt.legend()

In [ ]:
Npoles=30
print(ck)
exactpoles,exactamp=exactpoleres(sigma,mur*mu0,alpha,Npoles)
plt.stem(np.log10(np.abs(exactpoles[1:])), exactamp[1:],'rx',label="Exact")
plt.stem(np.log10(np.abs(xi[:])), ck[:,0]/(2*np.pi*alpha**3),'b',label=r"Approximate")
plt.xlabel(r"$\log_{10} (\xi_k)$")
plt.ylabel(r"$\tilde{c}_k/(2\pi \alpha^3) $")
#plt.savefig("results/sphere-eigngsolve/modesamp_exact_rep_P"+str(P)+".pdf")

In [ ]:
# Exact Step Function Response from a conducting non-permable sphere 
from scipy.optimize import fsolve
def derf(z):
    return 2/np.sqrt(np.pi)*np.exp(-z**2)
# Response from a step function
def exactA(t,sigma,mu,alpha,Npoles):
    mu0 = 4 * np.pi * (1e-7)
    mur=mu/mu0
    beta=np.sqrt(sigma*mu)*alpha
    x=np.sqrt(t)/beta
    T=t/beta**2
    # This is only valid for mur=1
    # Equation (11) Wait + Spies
    #A1=3*(T+1/3-2*np.sqrt(T/np.pi))
    # Method 1
    #for n in range(1,Npoles):
    #    derfn=derf(n/np.sqrt(T))
    #    erfcn=erfc(n/np.sqrt(T))
    #    A1+=-3*2*n*(np.sqrt(T)/n*derfn-2*erfcn)
    # Method 2 (not valid for small T)
    #A2=0.
    #for n in range(1,Npoles):
    #    A2+=6/np.pi**2*np.exp(-np.pi**2*n**2*T)/n**2 # eqn (22) Wait + Spies
    #   find solutions to fzerofun(deltan,mur) =0 

    # Only Method 2 works for permable case
    N=Npoles
    Deltan=fsolve(fzerofun, np.pi*np.linspace(1,N,N), args=(mur))
    Deltan=np.sort(Deltan)
    A2=-2*(mur-1)/(mur+2)
    for n in range(1,Npoles):
        deltan2=Deltan[n-1]**2
        A2+=6*mur/((mur+2)*(mur-1)+ deltan2)*np.exp(-deltan2*T) # eqn (18) Wait + Spies
    A1=A2
    return A1,A2


# Response from a impulse function
def exactB(t,sigma,mu,alpha,Npoles):
    beta=np.sqrt(sigma*mu)*alpha
    x=np.sqrt(t)/beta
    T=t/beta**2
    # Only Method 2 works for permable case
    N=Npoles
    Deltan=fsolve(fzerofun, np.pi*np.linspace(1,N,N), args=(mur))
    Deltan=np.sort(Deltan)
    B2=0.
    for n in range(1,Npoles):
        deltan2=Deltan[n-1]**2
        B2+=-6*mur/((mur+2)*(mur-1)+ deltan2)*deltan2*np.exp(-deltan2*T) # eqn (21) Wait + Spies
    B1=B2
    return B1,B2    

# Exact poles, amp for conducting permable sphere
def exactpoleres(sigma,mu,alpha,N):
    poles=np.zeros((N))
    amp=np.zeros((N))
    beta=np.sqrt(sigma*mu)*alpha
    mur=mu/mu0

#   find solutions to fzerofun(deltan,mur) =0 
    Deltan=fsolve(fzerofun, np.pi*np.linspace(1,N,N), args=(mur))
    Deltan=np.sort(Deltan)
    #print(Deltan)
    for n in range(1,N):
        # This is for mu=mu_0
        #deltan2=np.pi**2*n**2
        # This is for general mu
        deltan2=Deltan[n-1]**2
        poles[n]=-deltan2/beta**2
        # This is for mu =mu_0
        #amp[n]=6./(deltan2)
        # For general mu
        amp[n] = 6*mur/((mur+2)*(mur-1)+deltan2)
    return poles,amp

# Residual function to help find solutions to ((mur-1)+deltan2)*(tan deltan ) - (mur-1)*deltan = 0
def fzerofun(deltan,mur):
    return ((mur-1)+deltan**2)*np.tan(deltan) - (mur-1)*deltan

In [ ]:
time=Return_Dict['Time']
Sgn_step=Return_Dict['Sgn_step']
Sgn_impulse=Return_Dict['Sgn_impulse']

# Step response
Npoles=20
# Get the exact td step response (Method 2)
_, A2 = exactA(time, sigma, mur*mu0, alpha, Npoles)
l_td_exact_step = A2


plt.figure(1)
scale=1
plt.semilogx(time, l_td_exact_step /scale , 'b-', label="Exact Step")
scale=2*np.pi*alpha**3
for i in range(3):
    #scale=Sgn_step[0,i]
    plt.semilogx(time,Sgn_step[:,i]/scale,"r--",label="Numerical step, $\lambda_i$, $i$="+str(i+1))

plt.legend()
plt.xlabel(r"$t$ [s]")
plt.ylabel(r"$\lambda_i/(2\pi\alpha^3)$")
#plt.title("Step")
plt.grid(True, which="both")      

plt.figure(2)
scale=1
plt.semilogx(time, l_td_exact_step / scale, 'b-', label="Exact Step")
scale=2*np.pi*alpha**3
for i in range(3):
    plt.semilogx(time,Sgn_step[:,i]/scale,"r--",label="Numerical step, $\lambda_i$, $i$="+str(i+1))

plt.legend()
plt.xlabel(r"$t$ [s]")
plt.ylabel(r"$\lambda_i/(2\pi\alpha^3)$")
#plt.title("Step")
plt.grid(True, which="both") 

# Impulse response
# Get the exact td step response (Method 2)
_, B2 = exactB(time, sigma, mur*mu0, alpha, Npoles)
l_td_exact_impulse = B2


plt.figure(3)
scale=l_td_exact_impulse[0]
plt.loglog(time, l_td_exact_impulse / scale, 'b-', label="Exact Impulse")
for i in range(3):
    scale=Sgn_impulse[0,i]
    plt.loglog(time,Sgn_impulse[:,i]/scale,"r--",label=r"Numerical impulse, $\lambda_i$, $i$="+str(i+1))

plt.legend()
plt.xlabel(r"$t$ [s]")
plt.ylabel(r"Relative Response")
#plt.title("Impulse")
plt.grid(True, which="both")               

plt.figure(4)
scale=l_td_exact_impulse[0]
plt.semilogx(time, l_td_exact_impulse / scale, 'b-', label="Exact Impulse")
for i in range(3):
    scale=Sgn_impulse[0,i]
    plt.semilogx(time,Sgn_impulse[:,i]/scale,"r--",label=r"Numerical impulse, $\lambda_i$, i="+str(i+1))

plt.legend()
plt.xlabel(r"$t$ [s]")
plt.ylabel(r"Relative Response")
#plt.title("Impulse")
plt.grid(True, which="both") 
plt.show()